In [1]:
# install latest version from local files
%pip install -e ..

Obtaining file:///home/maarten/Documents/Projects/BioPAP/BioPAP
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for BioPAP (pyproject.toml) ... done
  Created wheel for BioPAP: filename=biopap-0.1.dev7+gf7272df3b.d20251110-0.editable-py3-none-any.whl size=3373 sha256=159a18cc16eea11e4c1740d6df70eddc97e101b151f47d1a2ab86952b3e3ea6c
  Stored in directory: /tmp/pip-ephem-wheel-cache-6g8iut6f/wheels/a9/55/80/ba98fe05e3d05116737c29a1c667eb5bffec478c135f508c7b
Successfully built BioPAP
  Attempting uninstall: BioPAP
    Found existing installation: BioPAP 0.1.dev7+gf7272df3b.d20251110
    Uninstalling BioPAP-0.1.dev7+gf7272df3b.d20251110:
      Successfully uninstalled BioPAP-0.1.dev7+gf7272df3b.d20251110
Note: you may need to restart the kernel to use updated packages.


In [2]:
from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.filtering import Filtering

### Loading data

In [3]:
bdf_file = '/data/BioPAP/data/CBN4IL230113.bdf'

In [4]:
eeg = LoadData(bdf_file)
eeg.raw_data.info

Extracting EDF parameters from /data/BioPAP/data/CBN4IL230113.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9601023  =      0.000 ...  4688.000 secs...


<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2013-01-23 14:42:49 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: CBN4IL230113>
>

### Anonymization

In [5]:
import random
daysback = random.randint(50, 150)
print(daysback)

117


In [6]:
ano = Anonymize(raw=eeg.raw_data, daysback=daysback, experimenter='J. Jansen', participant_id='AB00123')
# defaults: Participant='0'; Experimenter='mne_anonymize'
ano.processed_data.info

<Info | 10 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: False
 description: Anonymized using a time shift to preserve age at acquisition
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-09-28 14:42:49 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Rereferencing

In [7]:
ref = Rereference(raw=ano.processed_data, ref_channels=['EXG1', 'EXG2'])
ref.processed_data.info

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


<Info | 11 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-09-28 14:42:49 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

In [8]:
ref.ref_data

array([-0.01998071, -0.01997973, -0.01998345, ..., -0.01168142,
       -0.01168748, -0.01169515], shape=(9601024,))

### hEOG & vEOG

In [9]:
eog = EOGChannels(
    raw=ref.processed_data,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])
eog.processed_data.info

EEG channel type selected for re-referencing
Creating RawArray with float64 data, n_channels=2, n_times=9601024
    Range : 0 ... 9601023 =      0.000 ...  4688.000 secs
Ready.
Added the following bipolar channels:
hEOG, vEOG


<Info | 12 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 81 EEG, 1 Stimulus
 custom_ref_applied: True
 description: Anonymized using a time shift to preserve age at acquisition
 dig: 0 items
 experimenter: J. Jansen
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-09-28 14:42:49 UTC
 nchan: 82
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: AB00123>
>

### Filtering

In [ ]:
flt = Filtering(
    raw=eog.processed_data,
    l_freq=0.1,
    h_freq=35)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.10, 35.00 Hz: -6.02, -6.02 dB



In [ ]:
flt.processed